#Taller7 - Metodos Computacionales
SOFIA MOSCOSO ORTIZ 

In [1]:
import numpy as np
import sympy as sym
from scipy.special import factorial
from itertools import combinations
import scipy

El problema de diagonalización está relacionado al problema de encontrar la solución al sistema de ecuaciones de la forma: $$A\mathbf{x}=\lambda \mathbf{x},$$ donde $\lambda$ es un escalar, en este caso no podemos utilizar los métodos anteriores, pues $\mathbf{b}=\lambda \mathbf{x}$ también es desconocido.
Físicamente la diagonalización de una matriz se interpreta como una rotación de los ejes de la matriz de tal manera que estos queden alineados con sus autovectores (por ejemplo diagonalizar el tensor de inercia significa encontrar los ejes principales de rotación).

In [2]:
def row_lamb(i, lamb, A):
    M = 1.0*np.copy(A)
    M[[i]] *= lamb
    return M

def row_comb(i, j, lamb, A):
    M = 1.0*np.copy(A)
    M[[i]] += lamb * M[[j]]
    return M

def row_swap(i, j, A):
    M = 1.0*np.copy(A)
    M[[i, j]] = M[[j, i]]
    return M

In [3]:
#Gaussian Elimination
def Gaussian_Elimination( A0 ):
    #Making local copy of matrix
    A = np.copy(A0)
    #Detecting size of matrix
    n = len(A)

    #Sweeping all the columns in order to eliminate coefficients of the i-th variable
    for i in range( 0, n ):

        #Sweeping all the rows for the i-th column in order to find the first non-zero coefficient
        for j in range( i, n ):
            if A[i,j] != 0:
                #Normalization coefficient
                Norm = 1.0*A[i,j]
                break

        #Applying swap operation to put the non-zero coefficient in the i-th row
        A = row_swap( i, j, A )

        #Eliminating the coefficient associated to the i-th variable
        for j in range( i+1, n ):
            A = row_comb( j, i, -A[j,i]/Norm, A )

    #Normalizing n-th variable
    A = row_lamb( n-1, 1.0/A[n-1,n-1], A )

    #Finding solution
    x = np.zeros( n )
    x[n-1] = A[n-1,n]
    for i in range( n-1, -1, -1 ):
        x[i] = ( A[i,n] - sum(A[i,i+1:n]*x[i+1:n]) )/A[i,i]

    #Upper diagonal matrix and solutions x
    return A, x

In [4]:
def det(A):
    dt=0
    n = len(A)
    # -- Casos base --
    if  n ==1: dt = A[0,0]
    elif n ==2: dt = A[0][0]*A[1][1] - A[1][0]*A[0][1]
    # -- Forma general --
    else:
        for i in range(n):
            A_ij = np.delete(np.delete(A, 0, axis=1), i, axis=0) # A(0|j)
            dt += (-1)**(i) * A[i][0] * det( A_ij )
    return(dt)

#**1**
Consideremos la siguiente matriz:

$$
A={\begin{pmatrix}3&2\\7&-2\end{pmatrix}},
$$

Haga un rutina  en python para encontrar los autovalores y autovectores de una matrix dada, demuestre que $A^{-1}$ tiene autovalores $1/\lambda_i$ y los mismos autovectores de $A$. Use numpy para comprobar sus resultados. Use numpy para comprobar sus resultados.

In [14]:
def eigenValues(A):
    # (A - λIn)x = 0

    n = len(A)
    coef = []

    # -- coeficientes del polinomio caracteristico por método de menores --

    for k in  range(1, n): # menores de grado k, k ∈ {1, 2, ..., n}
        row_cols =  combinations( range(n), n-k) # filas y columnas a eliminar para encontrar los menores
                                                 # que resultan ser las combinaciones de filas-columnas posibles
        sum = 0

        for row_col in row_cols:
            M = np.copy(A)
            for I in row_col[::-1]:
                M = np.delete(np.delete(M, I, axis=1), I, axis=0) # Elimar la fila y columna I

            sum += det(M)       # Suma de menores de grado k

        coef.append(sum)

    coef.insert(0, -1)          # Coeficiente de grado de polinomio
    coef.append((-1**n)*det(A)) # Termino independiente
    coef = np.array(coef)[::-1]

    eigen_values = np.polynomial.polynomial.polyroots(coef)

    return eigen_values

def eigenVectors(A):
    n = len(A)
    vectores = []
    eigen_values = eigenValues(A)

    for lambd in eigen_values:
        M = A - lambd*np.identity(n)
        # Métodos intentados el domingo en la noche
        # x = scipy.linalg.solve(M, np.zeros((n,1)))
        x = np.linalg.lstsq(M, np.zeros((n,1)), rcond=None)[3]
        # x = GaussSeidel(M, np.zeros((n,1)))
        vectores.append(x)
    return vectores

In [15]:
np.zeros((1, 4))

array([[0., 0., 0., 0.]])

In [16]:
A = np.array([[3, 2] ,[7, -2]])

eigenValues(A), np.linalg.eig(A)[0]

(array([-4.,  5.]), array([ 5., -4.]))

#**2**
Haga una rutina que encuentre la inversa de una matriz y luego encuentre la solucion de las matrices $M_1$ y $M_2$.

**Nota** recuerde que la solucion de un sistema con el determinante se puede escribir como:
$$A^{-1}=\frac{1}{|A|} {\bf{Adj}}(A^T) $$
Por lo que debera creear una funcion que encuentre la ${Adj(A)}^T$

**Matrices de prueba**

$$M_1= \left[ \matrix{
2 & 1 & 3 &|+ 3 \\
4 & -1 & 3 &|-4 \\
-1 & 5 & 5 &|+ 9
}\right]  \\ $$

$$ M_2=\left[ \matrix{
3 & 1 & 3 & -4 &|1 \\
6 & 4 & 8 & -10 &|5 \\
3 & 2 & 5 & -1 &|6 \\
-9 & 5 & -2 & -4& |2
}\right] $$

In [17]:
def adj(A):
    n = len(A)
    M = np.zeros((n,n))
    for i in range(n):
        for j in range(n):
            M[i][j] = ((-1)**(i+j)) * det(np.delete(np.delete(A, i, axis=1), j, axis=0))

    return M

def inv(A):
    if det(A) != 0:
        return adj(A)* (1/det(A))
    else:
        raise Exception('Matrix is not invertible')

def sol_sistema(A, b):
    # Ax = b
    try:
        x = np.dot(inv(A), b)
    except:
        raise Exception('System has no solutions')

    return x


In [18]:
M1 = np.array([[2, 1, 3], [4, -1, 3], [1, 5, 5]])
b1 = np.array([[3], [-4], [9]])
M2 =  np.array([[3, 1, 3, -4], [6, 4, 8, -10], [3, 2, 5, -1], [-9, 5, -2, -4]])
b2 = np.array([[1], [5], [6], [2]])

In [19]:
sol_sistema(M1, b1)

array([[-7.66666667],
       [-4.16666667],
       [ 7.5       ]])

In [20]:
sol_sistema(M2, b2)

array([[0.70833333],
       [2.375     ],
       [0.        ],
       [0.875     ]])

#**3**
 Las llamadas matrices de Pauli, estan dadas por:


$$
σ_x={\begin{pmatrix}0&1\\1&0\end{pmatrix}},
$$

$$
σ_y={\begin{pmatrix}0&-i\\i&0\end{pmatrix}},
$$

$$
σ_z={\begin{pmatrix}1&0\\0&-1\end{pmatrix}},
$$

**A** Encuentre que $[σ_i,σ_j]\neq 0$ con $i,j= x,y, z$

**B** demuestre que ${σ_i}⁺={σ_i^*}^T=\sigma_i$ lo que significa que sus autovalores son reales.

**Nota** $[A,B]=AB-BA$


In [21]:
x = np.array([[0, 1], [1, 0]])
y = np.array([[0, -1j], [1j, 0]])
z = np.array([[1, 0], [0, -1]])
conmutador = lambda A, B: np.dot(A, B) - np.dot(B, A)

In [22]:
print(f'[σ_x, σ_y] = \n{conmutador(x, y)}\n\n[σ_x, σ_z] = \n{conmutador(x, z)}\n\n[σ_y, σ_z] =\n {conmutador(y,z)}')

[σ_x, σ_y] = 
[[0.+2.j 0.+0.j]
 [0.+0.j 0.-2.j]]

[σ_x, σ_z] = 
[[ 0 -2]
 [ 2  0]]

[σ_y, σ_z] =
 [[0.+0.j 0.+2.j]
 [0.+2.j 0.+0.j]]


In [23]:
eigenValues(x)

array([-1.,  1.])

In [25]:
eigenValues(y)

array([-1.+0.j,  1.+0.j])

In [26]:
eigenValues(z)

array([-1.,  1.])

#**4**
Repita el ejercicio anterior usando sympy, encuentre que en efecto sus autovalores son reales.

In [27]:
scipy.linalg.eig(x)[0]

array([ 1.+0.j, -1.+0.j])

In [28]:
scipy.linalg.eig(y)[0]

array([ 1.+0.j, -1.+0.j])

In [29]:
scipy.linalg.eig(z)[0]

array([ 1.+0.j, -1.+0.j])